# Agents in LlamaIndex

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies

We will install the dependencies for this unit.

In [ ]:
!pip install llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface -U -q

And, let's log in to Hugging Face to use serverless Inference APIs.

In [1]:
# from huggingface_hub import login

# login()
import os

gemini_token = os.getenv("GEMINI_KEY")

## Initialising agents

Let's start by initialising an agent. We will use the basic `AgentWorkflow` class to create an agent.

In [3]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream


def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b


llm = GoogleGenAI(model_name="models/gemini-3.1-flash-lite", api_key=gemini_token)

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[subtract, multiply, divide, add],
    llm=llm,
    system_prompt="You are a math agent that can add, subtract, multiply, and divide numbers using provided tools.",
)

Then, we can run the agent and get the response and reasoning behind the tool calls.

In [4]:
handler = agent.run("What is (2 + 2) * 2?")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp


Called tool:  add {'a': 2, 'b': 2} => 4

Called tool:  multiply {'b': 2, 'a': 4} => 8
The result of (2 + 2) * 2 is 8.

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'thought_signatures': [None], 'prompt_tokens': 468, 'completion_tokens': 16, 'total_tokens': 484}, blocks=[TextBlock(block_type='text', text='The result of (2 + 2) * 2 is 8.')]), structured_response=None, current_agent_name='Agent', raw={'content': {'parts': [{'media_resolution': None, 'code_execution_result': None, 'executable_code': None, 'file_data': None, 'function_call': None, 'function_response': None, 'inline_data': None, 'text': '', 'thought': None, 'thought_signature': None, 'video_metadata': None, 'tool_call': None, 'tool_response': None, 'part_metadata': None}], 'role': 'model'}, 'citation_metadata': None, 'finish_message': None, 'token_count': None, 'finish_reason': <FinishReason.STOP: 'STOP'>, 'grounding_metadata': None, 'avg_logprobs': None, 'index': 0, 'logprobs_result': None, 'safety_ratings': None, 'url_context_metadata': None, 'usage_metadata': {'cache_tokens_details': None,

In a similar fashion, we can pass state and context to the agent.


In [5]:
from llama_index.core.workflow import Context

ctx = Context(agent)

response = await agent.run("My name is Bob.", ctx=ctx)
response = await agent.run("What was my name again?", ctx=ctx)
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'thought_signatures': [None], 'prompt_tokens': 326, 'completion_tokens': 15, 'total_tokens': 380}, blocks=[TextBlock(block_type='text', text='Your name is Bob. How can I help you with any math today?')]), structured_response=None, current_agent_name='Agent', raw={'content': {'parts': [{'media_resolution': None, 'code_execution_result': None, 'executable_code': None, 'file_data': None, 'function_call': None, 'function_response': None, 'inline_data': None, 'text': '', 'thought': None, 'thought_signature': b'\x12\xa4\x02\n\xa1\x02\x01\x0c9\xd6\xc7\xb5\xe9[\x00\x07\xcd\x8d\xda\xef>\x97\x013\xa5\xbb8=\x1a\xa2\x8b\xf5\xbf\xc4\x92`V\xbd\x8b\x1bd\xb7\xc6\x05m\x93\xa8\xe7K\xd4o\xa1\x93\x92q\xfeJf]\xf3\xff&\x08\xc2\x9e\x02[\x9ei!A\x1c\r\xb5^bo5\x97\x83\x82\xfb[|\'\xb05\x9d\xb8\xdd#\x8d\xff\x1c\xaa\x0e\xd2\x8b;F\x0c\xdb\x12\xb4\xd5t\x99O\x17h\xb0\xb6\x1en\xbd\x90\xfd\x868\xa8mM\x0b\xb1\xa08P\x93\xa2\xd3

## Creating RAG Agents with QueryEngineTools

Let's now re-use the `QueryEngine` we defined in the [previous unit on tools](/tools.ipynb) and convert it into a `QueryEngineTool`. We will pass it to the `AgentWorkflow` class to create a RAG agent.

In [6]:
import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

# Create a vector store
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create a query engine
embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-2-preview", api_key=gemini_token)
llm = GoogleGenAI(model_name="models/gemini-3.1-flash-lite", api_key=gemini_token)
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm)
query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="personas",
    description="descriptions for various types of personas",
    return_direct=False,
)

# Create a RAG agent
query_engine_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[query_engine_tool],
    llm=llm,
    system_prompt="You are a helpful assistant that has access to a database containing persona descriptions. ",
)

And, we can once more get the response and reasoning behind the tool calls.

In [7]:
handler = query_engine_agent.run(
    "Search the database for 'science fiction' and return some persona descriptions."
)
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp


Called tool:  personas {'input': 'science fiction'} => There is no mention of science fiction, as the focus is on 19th-century American art and the local cultural heritage of Cincinnati.
I couldn't find any persona descriptions related to "science fiction" in the database. The current collection focuses primarily on **19th-century American art** and the **cultural heritage of Cincinnati**.

If you're interested in those topics, I can search for personas related to artists, historical figures from Cincinnati, or 19th-century culture instead!

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'thought_signatures': [None], 'prompt_tokens': 238, 'completion_tokens': 75, 'total_tokens': 664}, blocks=[TextBlock(block_type='text', text='I couldn\'t find any persona descriptions related to "science fiction" in the database. The current collection focuses primarily on **19th-century American art** and the **cultural heritage of Cincinnati**.\n\nIf you\'re interested in those topics, I can search for personas related to artists, historical figures from Cincinnati, or 19th-century culture instead!')]), structured_response=None, current_agent_name='Agent', raw={'content': {'parts': [{'media_resolution': None, 'code_execution_result': None, 'executable_code': None, 'file_data': None, 'function_call': None, 'function_response': None, 'inline_data': None, 'text': '', 'thought': None, 'thought_signature': b'\x12\xd6\x0f\n\xd3\x0f\x01\x0c9\xd6\xc7\x89\x7f\xf5\x85\xc6\xab\xb74|\xe4\xd2\x94\xc3\xe

## Creating multi-agent systems

We can also create multi-agent systems by passing multiple agents to the `AgentWorkflow` class.

In [8]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    ReActAgent,
)


# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


# Create agent configs
# NOTE: we can use FunctionAgent or ReActAgent here.
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = ReActAgent(
    name="info_lookup",
    description="Looks up information about XYZ",
    system_prompt="Use your tool to query a RAG system to answer information about XYZ",
    tools=[query_engine_tool],
    llm=llm,
)

# Create and run the workflow
agent = AgentWorkflow(agents=[calculator_agent, query_agent], root_agent="calculator")

# Run the system
handler = agent.run(user_msg="Can you add 5 and 3?")

In [9]:
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is English. I need to use the `add` tool to add 5 and 3.
Action: add
Action Input: {"a": 5, "b": 3}

Observation: 8
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: 5 plus 3 is 8.
Called tool:  add {'a': 5, 'b': 3} => 8
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: The sum of 5 and 3 is 8.

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'thought_signatures': [None], 'prompt_tokens': 849, 'completion_tokens': 37, 'total_tokens': 921}, blocks=[TextBlock(block_type='text', text='The sum of 5 and 3 is 8.')]), structured_response=None, current_agent_name='calculator', raw={'content': {'parts': [{'media_resolution': None, 'code_execution_result': None, 'executable_code': None, 'file_data': None, 'function_call': None, 'function_response': None, 'inline_data': None, 'text': '', 'thought': None, 'thought_signature': b"\x12\xd7\x01\n\xd4\x01\x01\x0c9\xd6\xc7{?\xaa\x16\x04\xe9\x14?qyy_1\x96\x82\x82\xad35vx\xd1\nb\xb7B\x19\x98\x16V\t\xf4\x97.\xcbu\xaa\xf9\xd0\x0b\xb4\n\xae\xb9\xb4q\xd7\x8eB\x06\x1d\xf4\x1e\x91\xe0\x8b\xf2\x00\x12\x19\x19\t.Ux\xfc8\xbf\xca\xf5\xf6\x02(H\xae\xab\xf2\x06\xfb\x07\x1d\xd9\xd8-\x92\xff\xeaA7\t\x07D#\x0c\xe9\x0fE\xc7\xfaG\x98\x1d>'\xd5t\xff\xb3\xa3IH\xecrysA?\xcb>\x8f\x9eOv\xaa\xe8\xd8\\7\xc6H{\xb4P\xac&\x80s